# Published model

Needs the `train` extra. The checkpoint and paper configuration are described in `MODEL_CARD.md`.

The cells below load the checkpoint and take a few actions on the synthetic demo catalogue with the paper observation layout. The Mission Candidate Sample is not included in the package. `make_paper_env()` needs a local copy; see `docs/CATALOGUE.md`.

In [ ]:
from aRieL.agents import RLAgentWrapper, load_default_model, make_checkpoint_smoke_env

env = make_checkpoint_smoke_env()
agent = RLAgentWrapper(load_default_model(), deterministic=True)
obs, info = env.reset(seed=42)
print("valid actions", int(info["action_mask"].sum()))

In [ ]:
total_reward = 0.0
for _ in range(8):
    action = agent.act(obs, info)
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += float(reward)
    if terminated or truncated:
        break

summary = info["mission_summary"]
print(f"steps {info['step_count']}  reward {total_reward:.3f}")
print(
    f"T1 {summary['tier1_completed']}  "
    f"T2 {summary['tier2_completed']}  "
    f"T3 {summary['tier3_completed']}"
)
env.close()

Optional: one full 3.5-year episode on the paper catalogue (seed 42). This needs `Ariel_MCS_Known_2025-08-18.csv` on `ARIEL_DATA` or in `./data/raw/`. It takes about a minute when the file is present.

In [ ]:
from aRieL.agents import run_inference_episode
from aRieL.data.load_catalogue import find_mcs_csv

if find_mcs_csv() is None:
    print("MCS not found. Set ARIEL_DATA or place MCS.csv in ./data/raw/. See docs/CATALOGUE.md.")
else:
    stats = run_inference_episode()
    print(stats.summary_str())